# Tandem Real vs AI Image Classifier

This notebook trains a **dual-branch model** on Hugging Face dataset **`ShreyashDhoot/Ai-vs_Real`**:

- **Deep CNN branch** for visual representation learning.
- **GLCM texture branch** that computes **12 texture features** across **8 directions** (total 96 features).

Both branches are fused exactly at the point where the CNN fully connected head begins.

In [ ]:
# If needed, uncomment this line and run once:
# !pip install -q torch torchvision datasets scikit-image scikit-learn tqdm

import math
import random
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from PIL import Image
from skimage.feature import graycomatrix
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
)
from tqdm.auto import tqdm

# -----------------------------
# Reproducibility and device
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -----------------------------
# Data configuration
# -----------------------------
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

# GLCM settings: 8 directions
GLCM_LEVELS = 64
GLCM_DISTANCES = [1]
GLCM_ANGLES = np.deg2rad([0, 22.5, 45, 67.5, 90, 112.5, 135, 157.5])
# 12 features x 8 directions = 96 dimensions
GLCM_FEATURE_DIM = 12 * len(GLCM_ANGLES)

In [ ]:
# -----------------------------
# Utility transforms (torch only)
# -----------------------------
def pil_to_chw_float(img: Image.Image, size: int = IMG_SIZE) -> torch.Tensor:
    img = img.convert("RGB").resize((size, size), Image.BILINEAR)
    arr = np.asarray(img, dtype=np.float32) / 255.0  # H, W, C in [0, 1]
    arr = np.transpose(arr, (2, 0, 1))  # C, H, W
    return torch.from_numpy(arr)


def normalize_tensor(x: torch.Tensor) -> torch.Tensor:
    # ImageNet-like normalization for stable CNN training
    mean = torch.tensor([0.485, 0.456, 0.406], dtype=x.dtype).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], dtype=x.dtype).view(3, 1, 1)
    return (x - mean) / std


def rgb_to_quantized_gray(
    img: Image.Image, size: int = IMG_SIZE, levels: int = GLCM_LEVELS
) -> np.ndarray:
    # Resize, convert to grayscale, and quantize into discrete levels for GLCM
    gray = img.convert("L").resize((size, size), Image.BILINEAR)
    arr = np.asarray(gray, dtype=np.uint8)
    q = (arr.astype(np.float32) / 256.0 * levels).astype(np.int32)
    q = np.clip(q, 0, levels - 1)
    return q


# -----------------------------
# 12 GLCM features per direction
# -----------------------------
def _glcm_features_from_matrix(P: np.ndarray) -> np.ndarray:
    """
    Compute 12 scalar features from one normalized GLCM matrix P.
    Features:
    1) contrast
    2) dissimilarity
    3) homogeneity
    4) ASM
    5) energy
    6) correlation
    7) entropy
    8) mean_i
    9) mean_j
    10) var_i
    11) cluster_shade
    12) cluster_prominence
    """
    eps = 1e-12
    P = P.astype(np.float64)
    P_sum = P.sum()
    if P_sum <= 0:
        return np.zeros(12, dtype=np.float32)
    P = P / (P_sum + eps)

    n = P.shape[0]
    i_idx = np.arange(n).reshape(-1, 1)
    j_idx = np.arange(n).reshape(1, -1)

    diff = i_idx - j_idx
    sum_ij = i_idx + j_idx

    contrast = np.sum((diff**2) * P)
    dissimilarity = np.sum(np.abs(diff) * P)
    homogeneity = np.sum(P / (1.0 + (diff**2)))

    asm = np.sum(P**2)
    energy = math.sqrt(max(asm, 0.0))

    p_i = P.sum(axis=1)
    p_j = P.sum(axis=0)

    mean_i = np.sum(i_idx[:, 0] * p_i)
    mean_j = np.sum(j_idx[0, :] * p_j)

    var_i = np.sum(((i_idx[:, 0] - mean_i) ** 2) * p_i)
    var_j = np.sum(((j_idx[0, :] - mean_j) ** 2) * p_j)

    std_i = math.sqrt(max(var_i, 0.0))
    std_j = math.sqrt(max(var_j, 0.0))

    corr_num = np.sum((i_idx - mean_i) * (j_idx - mean_j) * P)
    correlation = corr_num / (std_i * std_j + eps)

    entropy = -np.sum(P * np.log2(P + eps))

    centered_sum = sum_ij - mean_i - mean_j
    cluster_shade = np.sum((centered_sum**3) * P)
    cluster_prominence = np.sum((centered_sum**4) * P)

    feats = np.array(
        [
            contrast,
            dissimilarity,
            homogeneity,
            asm,
            energy,
            correlation,
            entropy,
            mean_i,
            mean_j,
            var_i,
            cluster_shade,
            cluster_prominence,
        ],
        dtype=np.float32,
    )
    return feats


def extract_glcm_96_features(img: Image.Image) -> np.ndarray:
    quant = rgb_to_quantized_gray(img, size=IMG_SIZE, levels=GLCM_LEVELS)
    glcm = graycomatrix(
        quant,
        distances=GLCM_DISTANCES,
        angles=GLCM_ANGLES,
        levels=GLCM_LEVELS,
        symmetric=True,
        normed=True,
    )

    # glcm shape: (levels, levels, num_distances=1, num_angles=8)
    all_feats: List[np.ndarray] = []
    for a in range(glcm.shape[3]):
        P = glcm[:, :, 0, a]
        all_feats.append(_glcm_features_from_matrix(P))

    feats = np.concatenate(all_feats, axis=0)  # (96,)
    return feats.astype(np.float32)

In [ ]:
# -----------------------------
# Forensic gradient/noise features
# -----------------------------
FORENSIC_FEATURE_DIM = 8


def extract_forensic_gradient_features(img: Image.Image, size: int = IMG_SIZE) -> np.ndarray:
    """
    Lightweight forensic descriptor inspired by gradient-field PCA + residual noise analysis.
    Returns 8 features:
      1) lambda1 (largest PCA eigenvalue of [gx, gy])
      2) lambda2 (smallest PCA eigenvalue)
      3) anisotropy = (lambda1-lambda2)/(lambda1+lambda2)
      4) gradient magnitude mean
      5) gradient magnitude std
      6) residual variance (high-frequency energy)
      7) residual kurtosis (peaky vs Gaussian-like tails)
      8) residual directional anisotropy from gradient covariance
    """
    gray = img.convert("L").resize((size, size), Image.BILINEAR)
    y = np.asarray(gray, dtype=np.float32) / 255.0

    gy, gx = np.gradient(y)
    g = np.stack([gx.reshape(-1), gy.reshape(-1)], axis=1)  # (N, 2)

    cov = np.cov(g, rowvar=False)
    eigvals = np.linalg.eigvalsh(cov)
    eigvals = np.sort(np.maximum(eigvals, 1e-12))[::-1]
    lam1, lam2 = float(eigvals[0]), float(eigvals[1])

    anis = (lam1 - lam2) / (lam1 + lam2 + 1e-12)
    mag = np.sqrt(gx * gx + gy * gy)
    mag_mean = float(np.mean(mag))
    mag_std = float(np.std(mag))

    # Fast high-pass residual (Laplacian-like) to approximate noise residual map.
    residual = y - (
        np.roll(y, 1, axis=0)
        + np.roll(y, -1, axis=0)
        + np.roll(y, 1, axis=1)
        + np.roll(y, -1, axis=1)
    ) / 4.0

    r = residual.reshape(-1).astype(np.float64)
    r_mean = float(np.mean(r))
    r_centered = r - r_mean
    r_var = float(np.mean(r_centered**2))
    r_fourth = float(np.mean(r_centered**4))
    r_kurt = r_fourth / (r_var * r_var + 1e-12)

    ry, rx = np.gradient(residual)
    rg = np.stack([rx.reshape(-1), ry.reshape(-1)], axis=1)
    rcov = np.cov(rg, rowvar=False)
    reig = np.linalg.eigvalsh(rcov)
    reig = np.sort(np.maximum(reig, 1e-12))[::-1]
    res_anis = float((reig[0] - reig[1]) / (reig[0] + reig[1] + 1e-12))

    feats = np.array(
        [lam1, lam2, anis, mag_mean, mag_std, r_var, r_kurt, res_anis],
        dtype=np.float32,
    )
    return feats


def compute_train_forensic_stats(hf_split, sample_size: int = 5000) -> Tuple[np.ndarray, np.ndarray]:
    """Estimate mean/std for forensic features from train split only."""
    n = len(hf_split)
    if n == 0:
        raise ValueError("Empty train split; cannot compute forensic normalization stats.")

    idxs = np.arange(n)
    if n > sample_size:
        rng = np.random.default_rng(SEED)
        idxs = rng.choice(idxs, size=sample_size, replace=False)

    feats = []
    for idx in tqdm(idxs, desc="Compute forensic stats", leave=False):
        ex = hf_split[int(idx)]
        img = ex["image"]
        if not isinstance(img, Image.Image):
            img = Image.fromarray(np.asarray(img))
        feats.append(extract_forensic_gradient_features(img))

    feats_np = np.stack(feats, axis=0).astype(np.float32)
    mean = feats_np.mean(axis=0)
    std = feats_np.std(axis=0)
    std = np.where(std < 1e-6, 1.0, std).astype(np.float32)
    return mean.astype(np.float32), std

In [ ]:
# -----------------------------
# HF dataset wrapper
# -----------------------------
def infer_label_column(ds_split) -> str:
    # Prefer conventional label keys if available
    for key in ["label", "labels", "target", "class"]:
        if key in ds_split.features:
            return key
    # Fallback: first integer-like column excluding image
    for key, feat in ds_split.features.items():
        if key == "image":
            continue
        if hasattr(feat, "num_classes") or "int" in str(feat):
            return key
    raise ValueError("Could not infer label column. Please set it manually.")


def compute_train_glcm_stats(hf_split, sample_size: int = 5000) -> Tuple[np.ndarray, np.ndarray]:
    """Estimate mean/std for 96-dim GLCM features from train split only."""
    n = len(hf_split)
    if n == 0:
        raise ValueError("Empty train split; cannot compute GLCM normalization stats.")

    idxs = np.arange(n)
    if n > sample_size:
        rng = np.random.default_rng(SEED)
        idxs = rng.choice(idxs, size=sample_size, replace=False)

    feats = []
    for idx in tqdm(idxs, desc="Compute GLCM stats", leave=False):
        ex = hf_split[int(idx)]
        img = ex["image"]
        if not isinstance(img, Image.Image):
            img = Image.fromarray(np.asarray(img))
        feats.append(extract_glcm_96_features(img))

    feats_np = np.stack(feats, axis=0).astype(np.float32)
    mean = feats_np.mean(axis=0)
    std = feats_np.std(axis=0)
    std = np.where(std < 1e-6, 1.0, std).astype(np.float32)
    return mean.astype(np.float32), std


class AIVsRealDataset(Dataset):
    def __init__(
        self,
        hf_split,
        label_col: str,
        augment: bool = False,
        glcm_mean: np.ndarray = None,
        glcm_std: np.ndarray = None,
        forensic_mean: np.ndarray = None,
        forensic_std: np.ndarray = None,
    ):
        self.ds = hf_split
        self.label_col = label_col
        self.augment = augment
        self.glcm_mean = glcm_mean
        self.glcm_std = glcm_std
        self.forensic_mean = forensic_mean
        self.forensic_std = forensic_std

    def __len__(self):
        return len(self.ds)

    def _maybe_augment(self, img: Image.Image) -> Image.Image:
        if not self.augment:
            return img
        # Light augmentations
        if random.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
        if random.random() < 0.2:
            img = img.rotate(random.uniform(-8, 8), resample=Image.BILINEAR)
        return img

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        ex = self.ds[idx]
        img = ex["image"]
        if not isinstance(img, Image.Image):
            img = Image.fromarray(np.asarray(img))

        img = self._maybe_augment(img)

        cnn_x = pil_to_chw_float(img)
        cnn_x = normalize_tensor(cnn_x)

        glcm_x = extract_glcm_96_features(img)
        if self.glcm_mean is not None and self.glcm_std is not None:
            glcm_x = (glcm_x - self.glcm_mean) / self.glcm_std

        forensic_x = extract_forensic_gradient_features(img)
        if self.forensic_mean is not None and self.forensic_std is not None:
            forensic_x = (forensic_x - self.forensic_mean) / self.forensic_std

        y = int(ex[self.label_col])

        return {
            "image": cnn_x,
            "glcm": torch.from_numpy(glcm_x.astype(np.float32)),
            "forensic": torch.from_numpy(forensic_x.astype(np.float32)),
            "label": torch.tensor(y, dtype=torch.long),
        }

In [ ]:
# -----------------------------
# Model: CNN + GLCM-DNN + Forensic fusion
# -----------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, dropout: float = 0.15):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout2d(dropout)
        self.pool = nn.MaxPool2d(2)

        self.shortcut = nn.Identity()
        if in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.act(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + identity  # Residual connection (minor accuracy/stability boost)
        out = self.act(out)
        out = self.dropout(out)
        out = self.pool(out)
        return out


class ResidualMLPBlock(nn.Module):
    def __init__(self, dim: int, dropout: float = 0.3):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.norm1 = nn.LayerNorm(dim)
        self.fc2 = nn.Linear(dim, dim)
        self.norm2 = nn.LayerNorm(dim)
        self.act = nn.ReLU(inplace=True)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        identity = x
        out = self.fc1(x)
        out = self.norm1(out)
        out = self.act(out)
        out = self.drop(out)
        out = self.fc2(out)
        out = self.norm2(out)
        out = out + identity
        out = self.act(out)
        return out


class CNNWithTextureFusion(nn.Module):
    def __init__(
        self,
        num_classes: int = 2,
        glcm_dim: int = GLCM_FEATURE_DIM,
        forensic_dim: int = FORENSIC_FEATURE_DIM,
    ):
        super().__init__()

        # Deep CNN feature extractor with residual conv blocks
        self.cnn_backbone = nn.Sequential(
            ConvBlock(3, 32, dropout=0.10),
            ConvBlock(32, 64, dropout=0.15),
            ConvBlock(64, 128, dropout=0.20),
            ConvBlock(128, 256, dropout=0.25),
            nn.Conv2d(256, 512, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.25),
            nn.AdaptiveAvgPool2d((1, 1)),
        )

        # Texture MLP branch
        self.texture_mlp = nn.Sequential(
            nn.Linear(glcm_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.30),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.30),
            ResidualMLPBlock(128, dropout=0.25),
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.ReLU(inplace=True),
        )

        # Forensic descriptor branch
        self.forensic_mlp = nn.Sequential(
            nn.Linear(forensic_dim, 64),
            nn.LayerNorm(64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(64, 32),
            nn.LayerNorm(32),
            nn.ReLU(inplace=True),
        )

        fusion_dim = 512 + 64 + 32

        # Fusion point: where CNN fully connected head starts
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.35),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.30),
            nn.Linear(128, num_classes),
        )

        # Projection head used by supervised contrastive objective.
        self.proj_head = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 128),
        )

    def encode_fused(self, image: torch.Tensor, glcm: torch.Tensor, forensic: torch.Tensor) -> torch.Tensor:
        cnn_feat = self.cnn_backbone(image)
        cnn_feat = cnn_feat.flatten(1)  # (B, 512)

        tex_feat = self.texture_mlp(glcm)  # (B, 64)
        forensic_feat = self.forensic_mlp(forensic)  # (B, 32)

        fused = torch.cat([cnn_feat, tex_feat, forensic_feat], dim=1)
        return fused

    def forward(
        self,
        image: torch.Tensor,
        glcm: torch.Tensor,
        forensic: torch.Tensor,
        return_features: bool = False,
    ):
        fused = self.encode_fused(image, glcm, forensic)
        logits = self.classifier(fused)

        if return_features:
            z = self.proj_head(fused)
            return logits, z

        return logits


def count_parameters(model: nn.Module) -> Dict[str, int]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable = total - trainable
    return {
        "total": total,
        "trainable": trainable,
        "non_trainable": non_trainable,
    }

In [ ]:
# -----------------------------
# Train / eval helpers
# -----------------------------
@dataclass
class TrainConfig:
    lr: float = 7e-4
    epochs: int = 12
    weight_decay: float = 2e-4
    focal_gamma: float = 2.0
    patch_weight: float = 0.2
    contrast_weight: float = 0.1
    contrast_temp: float = 0.07
    patch_min_scale: float = 0.55
    patch_max_scale: float = 0.85


class FocalLoss(nn.Module):
    def __init__(self, alpha: torch.Tensor = None, gamma: float = 2.0, reduction: str = "mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction="none")
        pt = torch.exp(-ce)
        loss = ((1.0 - pt) ** self.gamma) * ce

        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss


class SupervisedContrastiveLoss(nn.Module):
    def __init__(self, temperature: float = 0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        """
        features: (B, D), labels: (B,)
        """
        if features.ndim != 2:
            raise ValueError("SupervisedContrastiveLoss expects features shape (B, D).")

        features = F.normalize(features, dim=1)
        labels = labels.contiguous().view(-1, 1)

        mask = torch.eq(labels, labels.T).float().to(features.device)

        logits = torch.matmul(features, features.T) / self.temperature
        logits = logits - logits.max(dim=1, keepdim=True).values.detach()

        self_mask = torch.eye(features.size(0), device=features.device)
        mask = mask * (1.0 - self_mask)

        exp_logits = torch.exp(logits) * (1.0 - self_mask)
        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-12)

        mask_sum = mask.sum(dim=1)
        valid = mask_sum > 0
        if valid.sum() == 0:
            return features.new_tensor(0.0)

        mean_log_prob_pos = (mask * log_prob).sum(dim=1) / (mask_sum + 1e-12)
        loss = -mean_log_prob_pos[valid].mean()
        return loss


def make_random_patch_view(
    x_img: torch.Tensor,
    min_scale: float = 0.55,
    max_scale: float = 0.85,
) -> torch.Tensor:
    """Create a resized random patch view per sample for consistency regularization."""
    b, c, h, w = x_img.shape
    out = []
    for i in range(b):
        scale = random.uniform(min_scale, max_scale)
        ph = max(16, int(h * scale))
        pw = max(16, int(w * scale))
        top = 0 if ph >= h else random.randint(0, h - ph)
        left = 0 if pw >= w else random.randint(0, w - pw)

        patch = x_img[i : i + 1, :, top : top + ph, left : left + pw]
        patch = F.interpolate(patch, size=(h, w), mode="bilinear", align_corners=False)
        out.append(patch)

    return torch.cat(out, dim=0)


def _compute_precision_recall(y_true: np.ndarray, y_pred: np.ndarray) -> Tuple[float, float]:
    """Binary precision/recall when possible, otherwise macro-averaged for multiclass."""
    unique_classes = np.unique(y_true)
    if unique_classes.size <= 2:
        precision = float(precision_score(y_true, y_pred, zero_division=0))
        recall = float(recall_score(y_true, y_pred, zero_division=0))
    else:
        precision = float(precision_score(y_true, y_pred, average="macro", zero_division=0))
        recall = float(recall_score(y_true, y_pred, average="macro", zero_division=0))
    return precision, recall


def run_epoch(
    model,
    loader,
    criterion,
    optimizer=None,
    use_patch_consistency: bool = False,
    use_contrastive: bool = False,
    patch_weight: float = 0.2,
    contrast_weight: float = 0.1,
    patch_min_scale: float = 0.55,
    patch_max_scale: float = 0.85,
    contrastive_criterion=None,
):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    all_targets = []
    all_preds = []

    pbar = tqdm(loader, desc="Train" if train_mode else "Eval", leave=False)
    for batch in pbar:
        x_img = batch["image"].to(device, non_blocking=True)
        x_glcm = batch["glcm"].to(device, non_blocking=True)
        x_forensic = batch["forensic"].to(device, non_blocking=True)
        y = batch["label"].to(device, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            if train_mode and use_contrastive:
                logits, z = model(x_img, x_glcm, x_forensic, return_features=True)
            else:
                logits = model(x_img, x_glcm, x_forensic)
                z = None

            cls_loss = criterion(logits, y)
            loss = cls_loss

            if train_mode and use_patch_consistency:
                x_patch = make_random_patch_view(
                    x_img,
                    min_scale=patch_min_scale,
                    max_scale=patch_max_scale,
                )
                patch_logits = model(x_patch, x_glcm, x_forensic)

                p = F.log_softmax(logits, dim=1)
                q = F.softmax(patch_logits, dim=1)
                patch_loss = F.kl_div(p, q, reduction="batchmean")
                loss = loss + patch_weight * patch_loss

            if train_mode and use_contrastive:
                if contrastive_criterion is None:
                    raise ValueError("contrastive_criterion is required when use_contrastive=True")
                contrast_loss = contrastive_criterion(z, y)
                loss = loss + contrast_weight * contrast_loss

            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
                optimizer.step()

        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == y).sum().item()
        total_samples += y.size(0)
        total_loss += loss.item() * y.size(0)

        all_targets.append(y.detach().cpu())
        all_preds.append(preds.detach().cpu())

        pbar.set_postfix(
            loss=f"{(total_loss / max(total_samples, 1)):.4f}",
            acc=f"{(total_correct / max(total_samples, 1)):.4f}",
        )

    avg_loss = total_loss / max(total_samples, 1)
    avg_acc = total_correct / max(total_samples, 1)

    y_true = torch.cat(all_targets).numpy() if all_targets else np.array([], dtype=np.int64)
    y_pred = torch.cat(all_preds).numpy() if all_preds else np.array([], dtype=np.int64)

    if y_true.size == 0:
        avg_precision, avg_recall = 0.0, 0.0
    else:
        avg_precision, avg_recall = _compute_precision_recall(y_true, y_pred)

    return avg_loss, avg_acc, avg_precision, avg_recall


def predict_probs(model, loader):
    """Return y_true and positive-class probabilities for binary threshold tuning."""
    model.eval()
    ys, probs = [], []
    with torch.no_grad():
        for batch in loader:
            x_img = batch["image"].to(device)
            x_glcm = batch["glcm"].to(device)
            x_forensic = batch["forensic"].to(device)
            y = batch["label"].to(device)
            logits = model(x_img, x_glcm, x_forensic)

            if logits.shape[1] != 2:
                raise ValueError("Threshold tuning currently expects binary classification with 2 logits.")

            pos_prob = torch.softmax(logits, dim=1)[:, 1]
            ys.append(y.cpu().numpy())
            probs.append(pos_prob.cpu().numpy())

    return np.concatenate(ys), np.concatenate(probs)


def tune_threshold_for_f1(y_true: np.ndarray, y_prob: np.ndarray) -> Tuple[float, float, float, float]:
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)

    # precision_recall_curve returns len(thresholds) = len(precision) - 1
    f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    best_idx = int(np.argmax(f1_scores))

    best_threshold = float(thresholds[best_idx])
    best_precision = float(precision[best_idx])
    best_recall = float(recall[best_idx])
    best_f1 = float(f1_scores[best_idx])

    return best_threshold, best_precision, best_recall, best_f1


def evaluate_binary_metrics(model, val_loader, test_loader):
    """Tune threshold on val, then report F1 and PR-AUC on test."""
    y_val_true, y_val_prob = predict_probs(model, val_loader)
    best_threshold, _, _, _ = tune_threshold_for_f1(y_val_true, y_val_prob)

    y_test_true, y_test_prob = predict_probs(model, test_loader)
    y_test_pred = (y_test_prob >= best_threshold).astype(np.int64)

    pr_auc = float(average_precision_score(y_test_true, y_test_prob))
    f1 = float(f1_score(y_test_true, y_test_pred, zero_division=0))

    return {
        "threshold": float(best_threshold),
        "f1": f1,
        "pr_auc": pr_auc,
    }

In [ ]:
# -----------------------------
# Load HF dataset and build splits
# -----------------------------
raw = load_dataset("ShreyashDhoot/Ai-vs_Real")
print(raw)

# Find label column from first available split
first_split = next(iter(raw.keys()))
label_col = infer_label_column(raw[first_split])
print(f"Detected label column: {label_col}")

# Standardize split availability
if "train" in raw and "validation" in raw and "test" in raw:
    ds_train = raw["train"]
    ds_val = raw["validation"]
    ds_test = raw["test"]
elif "train" in raw and "test" in raw:
    # Create validation from train
    split = raw["train"].train_test_split(test_size=0.1, seed=SEED)
    ds_train = split["train"]
    ds_val = split["test"]
    ds_test = raw["test"]
elif "train" in raw:
    # Create both val and test from train
    split1 = raw["train"].train_test_split(test_size=0.2, seed=SEED)
    split2 = split1["test"].train_test_split(test_size=0.5, seed=SEED)
    ds_train = split1["train"]
    ds_val = split2["train"]
    ds_test = split2["test"]
else:
    # Fallback: use first split and create train/val/test
    split1 = raw[first_split].train_test_split(test_size=0.2, seed=SEED)
    split2 = split1["test"].train_test_split(test_size=0.5, seed=SEED)
    ds_train = split1["train"]
    ds_val = split2["train"]
    ds_test = split2["test"]

print(f"Train: {len(ds_train)} | Val: {len(ds_val)} | Test: {len(ds_test)}")

# Infer number of classes
try:
    feat = ds_train.features[label_col]
    num_classes = getattr(feat, "num_classes", None) or len(set(ds_train[label_col]))
except Exception:
    num_classes = len(set(ds_train[label_col]))

print(f"Num classes: {num_classes}")

# -----------------------------
# Check class imbalance and build class weights
# -----------------------------
train_labels = np.array(ds_train[label_col], dtype=np.int64)
class_counts = np.bincount(train_labels, minlength=num_classes)
class_ratio = class_counts / max(class_counts.sum(), 1)
imbalance_ratio = class_counts.max() / max(class_counts.min(), 1)

# Inverse-frequency weighting with mean-normalization for stable scale
class_weights_np = (class_counts.sum() / np.maximum(class_counts, 1)).astype(np.float32)
class_weights_np = class_weights_np / class_weights_np.mean()
class_weights = torch.tensor(class_weights_np, dtype=torch.float32, device=device)

print("Class counts:", class_counts.tolist())
print("Class ratio:", [round(float(r), 4) for r in class_ratio])
print(f"Imbalance ratio (max/min): {imbalance_ratio:.3f}")
print("Class weights:", [round(float(w), 4) for w in class_weights.cpu()])

# -----------------------------
# Normalize handcrafted features (fit on train only)
# -----------------------------
glcm_mean, glcm_std = compute_train_glcm_stats(ds_train, sample_size=5000)
forensic_mean, forensic_std = compute_train_forensic_stats(ds_train, sample_size=5000)
print("GLCM and forensic mean/std computed from train split.")

train_ds = AIVsRealDataset(
    ds_train,
    label_col=label_col,
    augment=True,
    glcm_mean=glcm_mean,
    glcm_std=glcm_std,
    forensic_mean=forensic_mean,
    forensic_std=forensic_std,
)
val_ds = AIVsRealDataset(
    ds_val,
    label_col=label_col,
    augment=False,
    glcm_mean=glcm_mean,
    glcm_std=glcm_std,
    forensic_mean=forensic_mean,
    forensic_std=forensic_std,
)
test_ds = AIVsRealDataset(
    ds_test,
    label_col=label_col,
    augment=False,
    glcm_mean=glcm_mean,
    glcm_std=glcm_std,
    forensic_mean=forensic_mean,
    forensic_std=forensic_std,
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

In [ ]:
# -----------------------------
# Ablation config and experiment setup
# -----------------------------
cfg = TrainConfig(
    lr=7e-4,
    epochs=12,
    weight_decay=2e-4,
    focal_gamma=2.0,
    patch_weight=0.2,      # small patch-consistency weight
    contrast_weight=0.1,   # small supervised-contrastive weight
    contrast_temp=0.07,
)

ablation_settings = [
    {"name": "baseline", "patch": False, "contrast": False},
    {"name": "baseline+patch", "patch": True, "contrast": False},
    {"name": "baseline+contrast", "patch": False, "contrast": True},
    {"name": "baseline+both", "patch": True, "contrast": True},
]

print("Ablation settings:")
for setting in ablation_settings:
    print(f"- {setting['name']}")

example_model = CNNWithTextureFusion(num_classes=num_classes, glcm_dim=GLCM_FEATURE_DIM).to(device)
param_stats = count_parameters(example_model)
print("\nParameter details (single model instance):")
print(f"Total parameters:      {param_stats['total']:,}")
print(f"Trainable parameters:  {param_stats['trainable']:,}")
print(f"Non-trainable params:  {param_stats['non_trainable']:,}")
del example_model

print(f"\nBase loss: FocalLoss(gamma={cfg.focal_gamma}) with class weights")
print(f"Patch weight: {cfg.patch_weight}")
print(f"Contrastive weight: {cfg.contrast_weight} (temp={cfg.contrast_temp})")

In [ ]:
# -----------------------------
# Persistent training utilities (run once)
# -----------------------------
import os
import json

CHECKPOINT_DIR = "checkpoints_tandem"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

training_plan = [
    {"run_id": "model_1_baseline", "setting": {"name": "baseline", "patch": False, "contrast": False}},
    {"run_id": "model_2_patch", "setting": {"name": "baseline+patch", "patch": True, "contrast": False}},
    {"run_id": "model_3_contrast", "setting": {"name": "baseline+contrast", "patch": False, "contrast": True}},
]


def _paths_for_run(run_id: str):
    weight_path = os.path.join(CHECKPOINT_DIR, f"{run_id}_best.pth")
    history_path = os.path.join(CHECKPOINT_DIR, f"{run_id}_history.json")
    summary_path = os.path.join(CHECKPOINT_DIR, f"{run_id}_summary.json")
    return weight_path, history_path, summary_path


def train_one_run(run_id: str, setting: dict, cfg_obj: TrainConfig):
    """
    Train one experiment and save artifacts to disk.
    Safe to run independently in separate cells.
    """
    weight_path, history_path, summary_path = _paths_for_run(run_id)

    print("\n" + "=" * 72)
    print(f"Run ID: {run_id} | Experiment: {setting['name']}")
    print("=" * 72)

    model_local = CNNWithTextureFusion(num_classes=num_classes, glcm_dim=GLCM_FEATURE_DIM).to(device)
    optimizer = torch.optim.AdamW(model_local.parameters(), lr=cfg_obj.lr, weight_decay=cfg_obj.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
    )

    criterion = FocalLoss(alpha=class_weights, gamma=cfg_obj.focal_gamma)
    contrastive_criterion = SupervisedContrastiveLoss(temperature=cfg_obj.contrast_temp)

    best_val_acc = -1.0
    epoch_history = []

    for epoch in range(1, cfg_obj.epochs + 1):
        tr_loss, tr_acc, tr_prec, tr_rec = run_epoch(
            model_local,
            train_loader,
            criterion=criterion,
            optimizer=optimizer,
            use_patch_consistency=setting["patch"],
            use_contrastive=setting["contrast"],
            patch_weight=cfg_obj.patch_weight,
            contrast_weight=cfg_obj.contrast_weight,
            patch_min_scale=cfg_obj.patch_min_scale,
            patch_max_scale=cfg_obj.patch_max_scale,
            contrastive_criterion=contrastive_criterion,
        )
        va_loss, va_acc, va_prec, va_rec = run_epoch(
            model_local,
            val_loader,
            criterion=criterion,
            optimizer=None,
        )

        scheduler.step(va_loss)
        lr_now = float(optimizer.param_groups[0]["lr"])

        epoch_row = {
            "epoch": int(epoch),
            "train_loss": float(tr_loss),
            "train_acc": float(tr_acc),
            "train_precision": float(tr_prec),
            "train_recall": float(tr_rec),
            "val_loss": float(va_loss),
            "val_acc": float(va_acc),
            "val_precision": float(va_prec),
            "val_recall": float(va_rec),
            "lr": lr_now,
        }
        epoch_history.append(epoch_row)

        print(
            f"Epoch {epoch:02d}/{cfg_obj.epochs} | "
            f"Train Loss: {tr_loss:.4f}, Train Acc: {tr_acc:.4f}, Train Prec: {tr_prec:.4f}, Train Rec: {tr_rec:.4f} | "
            f"Val Loss: {va_loss:.4f}, Val Acc: {va_acc:.4f}, Val Prec: {va_prec:.4f}, Val Rec: {va_rec:.4f} | "
            f"LR: {lr_now:.2e}"
        )

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save(model_local.state_dict(), weight_path)
            with open(history_path, "w", encoding="utf-8") as f:
                json.dump(epoch_history, f, indent=2)
            print(f"  Saved improved checkpoint -> {weight_path}")

    if not os.path.exists(weight_path):
        torch.save(model_local.state_dict(), weight_path)

    best_model = CNNWithTextureFusion(num_classes=num_classes, glcm_dim=GLCM_FEATURE_DIM).to(device)
    best_model.load_state_dict(torch.load(weight_path, map_location=device))
    best_model.eval()

    metrics = evaluate_binary_metrics(best_model, val_loader, test_loader)

    cross_domain_f1 = None
    cross_domain_pr_auc = None
    if "cross_domain_loader" in globals() and cross_domain_loader is not None:
        cross_metrics = evaluate_binary_metrics(best_model, val_loader, cross_domain_loader)
        cross_domain_f1 = cross_metrics["f1"]
        cross_domain_pr_auc = cross_metrics["pr_auc"]

    summary = {
        "run_id": run_id,
        "experiment": setting["name"],
        "use_patch": bool(setting["patch"]),
        "use_contrast": bool(setting["contrast"]),
        "best_val_acc": float(best_val_acc),
        "test_f1": float(metrics["f1"]),
        "test_pr_auc": float(metrics["pr_auc"]),
        "threshold": float(metrics["threshold"]),
        "cross_domain_f1": None if cross_domain_f1 is None else float(cross_domain_f1),
        "cross_domain_pr_auc": None if cross_domain_pr_auc is None else float(cross_domain_pr_auc),
        "weight_path": weight_path,
        "history_path": history_path,
    }

    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(
        f"Result | F1: {summary['test_f1']:.4f} | PR-AUC: {summary['test_pr_auc']:.4f} | "
        f"Threshold: {summary['threshold']:.4f}"
    )
    print(f"Saved run summary -> {summary_path}")

    return summary


def load_run_summary(run_id: str):
    _, _, summary_path = _paths_for_run(run_id)
    if not os.path.exists(summary_path):
        raise FileNotFoundError(f"Summary not found for {run_id}. Run its training cell first.")
    with open(summary_path, "r", encoding="utf-8") as f:
        return json.load(f)


print("Training utilities ready.")
print("Run Cell 11, then Cell 12, then Cell 13 independently.")
print(f"Checkpoints directory: {CHECKPOINT_DIR}")

In [ ]:
# -----------------------------
# Train model 1/3: baseline
# -----------------------------
run_1 = training_plan[0]
summary_model_1 = train_one_run(run_1["run_id"], run_1["setting"], cfg)
summary_model_1

In [ ]:
# -----------------------------
# Train model 2/3: baseline+patch
# -----------------------------
run_2 = training_plan[1]
summary_model_2 = train_one_run(run_2["run_id"], run_2["setting"], cfg)
summary_model_2

In [ ]:
# -----------------------------
# Train model 3/3: baseline+contrast
# -----------------------------
run_3 = training_plan[2]
summary_model_3 = train_one_run(run_3["run_id"], run_3["setting"], cfg)
summary_model_3

In [ ]:
# -----------------------------
# Aggregate saved runs + load best model into `model`
# -----------------------------
saved_results = []
for run in training_plan:
    try:
        saved_results.append(load_run_summary(run["run_id"]))
    except FileNotFoundError:
        print(f"Missing run summary for {run['run_id']} (run its training cell first).")

if len(saved_results) == 0:
    raise RuntimeError("No saved runs found yet. Train at least one model cell first.")

print("\n" + "#" * 72)
print("Saved run summaries")
print("#" * 72)
for row in saved_results:
    print(
        f"{row['run_id']:<18} | {row['experiment']:<18} | "
        f"Val Acc: {row['best_val_acc']:.4f} | "
        f"Test F1: {row['test_f1']:.4f} | "
        f"Test PR-AUC: {row['test_pr_auc']:.4f}"
    )

best_overall = max(saved_results, key=lambda r: r["test_f1"])
print("\nBest run:", best_overall["run_id"], "|", best_overall["experiment"])
print(
    f"Best Test F1: {best_overall['test_f1']:.4f}, "
    f"Best PR-AUC: {best_overall['test_pr_auc']:.4f}, "
    f"Threshold: {best_overall['threshold']:.4f}"
)

model = CNNWithTextureFusion(num_classes=num_classes, glcm_dim=GLCM_FEATURE_DIM).to(device)
model.load_state_dict(torch.load(best_overall["weight_path"], map_location=device))
model.eval()
print(f"Loaded best model from: {best_overall['weight_path']}")

In [ ]:
# -----------------------------
# Detailed report for loaded best model
# -----------------------------
if "model" not in globals() or model is None:
    recovered_results = []
    for run in training_plan:
        try:
            recovered_results.append(load_run_summary(run["run_id"]))
        except FileNotFoundError:
            pass

    if len(recovered_results) == 0:
        raise RuntimeError("No trained model found. Run at least one model training cell first.")

    best_recovered = max(recovered_results, key=lambda r: r["test_f1"])
    model = CNNWithTextureFusion(num_classes=num_classes, glcm_dim=GLCM_FEATURE_DIM).to(device)
    model.load_state_dict(torch.load(best_recovered["weight_path"], map_location=device))
    model.eval()
    print(f"Recovered and loaded best model from: {best_recovered['weight_path']}")

y_val_true, y_val_prob = predict_probs(model, val_loader)
best_threshold, val_p, val_r, val_f1 = tune_threshold_for_f1(y_val_true, y_val_prob)

y_test_true, y_test_prob = predict_probs(model, test_loader)
y_test_pred = (y_test_prob >= best_threshold).astype(np.int64)

print("\nBest validation threshold (F1-based):")
print(f"Threshold: {best_threshold:.4f} | Precision: {val_p:.4f} | Recall: {val_r:.4f} | F1: {val_f1:.4f}")

print("\nConfusion Matrix (threshold-tuned):")
print(confusion_matrix(y_test_true, y_test_pred))

print("\nClassification Report (threshold-tuned):")
print(classification_report(y_test_true, y_test_pred, digits=4))

# Precision-recall focused metrics
pr_auc = average_precision_score(y_test_true, y_test_prob)
precision_t = precision_score(y_test_true, y_test_pred, zero_division=0)
recall_t = recall_score(y_test_true, y_test_pred, zero_division=0)
f1_t = f1_score(y_test_true, y_test_pred, zero_division=0)

print("\nPR-focused metrics:")
print(f"PR-AUC (Average Precision): {pr_auc:.4f}")
print(f"Precision @ tuned threshold: {precision_t:.4f}")
print(f"Recall @ tuned threshold:    {recall_t:.4f}")
print(f"F1 @ tuned threshold:        {f1_t:.4f}")

# Optional cross-domain summary for best model
if "cross_domain_loader" in globals() and cross_domain_loader is not None:
    cross_m = evaluate_binary_metrics(model, val_loader, cross_domain_loader)
    print("\nCross-domain metrics (best model):")
    print(f"Cross-domain PR-AUC: {cross_m['pr_auc']:.4f}")
    print(f"Cross-domain F1:     {cross_m['f1']:.4f}")

# Precision-Recall curve
precision_curve, recall_curve, _ = precision_recall_curve(y_test_true, y_test_prob)
plt.figure(figsize=(6, 5))
plt.plot(recall_curve, precision_curve, linewidth=2)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision-Recall Curve (AP={pr_auc:.4f})")
plt.grid(alpha=0.3)
plt.show()

save_path = "best_loaded_model_for_report.pth"
torch.save(model.state_dict(), save_path)
print(f"Saved current loaded model weights to: {save_path}")

In [ ]:
# -----------------------------
# Layer-wise heatmaps (1 AI + 1 Real) for loaded tandem model
# -----------------------------


def _get_label_name_map(hf_split, label_col_name: str):
    feat = hf_split.features.get(label_col_name, None)
    names = getattr(feat, "names", None)
    if names is not None:
        return {i: str(n) for i, n in enumerate(names)}
    return {0: "AI", 1: "Real"}


def _find_first_index_for_label(hf_split, label_col_name: str, target_label: int):
    labels = hf_split[label_col_name]
    for idx, y in enumerate(labels):
        if int(y) == int(target_label):
            return idx
    raise ValueError(f"No sample found for label={target_label} in ds_test")


def _prepare_tandem_inputs(pil_img: Image.Image):
    # Match training preprocessing exactly
    x_img = pil_to_chw_float(pil_img)
    x_img = normalize_tensor(x_img)

    glcm_x = extract_glcm_96_features(pil_img)
    if glcm_mean is not None and glcm_std is not None:
        glcm_x = (glcm_x - glcm_mean) / glcm_std

    forensic_x = extract_forensic_gradient_features(pil_img)
    if forensic_mean is not None and forensic_std is not None:
        forensic_x = (forensic_x - forensic_mean) / forensic_std

    x_img = x_img.unsqueeze(0).to(device)
    x_glcm = torch.from_numpy(glcm_x.astype(np.float32)).unsqueeze(0).to(device)
    x_forensic = torch.from_numpy(forensic_x.astype(np.float32)).unsqueeze(0).to(device)
    return x_img, x_glcm, x_forensic


def _collect_cnn_conv_activations(model_obj, x_img, x_glcm, x_forensic):
    activations = {}
    hooks = []

    # Collect all Conv2d layers in CNN branch for true layer-wise maps.
    conv_layer_names = [
        name for name, module in model_obj.named_modules()
        if name.startswith("cnn_backbone") and isinstance(module, nn.Conv2d)
    ]

    for name, module in model_obj.named_modules():
        if name in conv_layer_names:
            def _hook_fn(_, __, output, layer_name=name):
                activations[layer_name] = output.detach().cpu()
            hooks.append(module.register_forward_hook(_hook_fn))

    with torch.no_grad():
        logits = model_obj(x_img, x_glcm, x_forensic)
        prob_real = float(torch.softmax(logits, dim=1)[0, 1].item())

    for h in hooks:
        h.remove()

    return activations, prob_real


def _activation_to_heatmap(act_tensor: torch.Tensor, out_size: int = IMG_SIZE) -> np.ndarray:
    # act_tensor expected shape: (1, C, H, W)
    fmap = act_tensor[0].abs().mean(dim=0).numpy()
    fmap = fmap - fmap.min()
    if fmap.max() > 0:
        fmap = fmap / fmap.max()

    fmap_t = torch.from_numpy(fmap).unsqueeze(0).unsqueeze(0).float()
    fmap_t = F.interpolate(fmap_t, size=(out_size, out_size), mode="bilinear", align_corners=False)
    return fmap_t.squeeze().numpy()


def _plot_layerwise_heatmaps_for_one_sample(model_obj, pil_img: Image.Image, true_label: int, label_name_map: dict):
    x_img, x_glcm, x_forensic = _prepare_tandem_inputs(pil_img)
    activations, prob_real = _collect_cnn_conv_activations(model_obj, x_img, x_glcm, x_forensic)

    pred_label = int(prob_real >= 0.5)

    img_np = np.asarray(pil_img.convert("RGB").resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32) / 255.0

    layer_names = list(activations.keys())
    n_layers = len(layer_names)
    n_cols = 4
    n_plots = n_layers + 1  # original + heatmaps
    n_rows = int(np.ceil(n_plots / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.2 * n_rows))
    axes = np.array(axes).reshape(-1)

    axes[0].imshow(img_np)
    axes[0].set_title(
        f"Original\nTrue: {label_name_map.get(true_label, true_label)} | "
        f"Pred: {label_name_map.get(pred_label, pred_label)} ({prob_real:.3f})"
    )
    axes[0].axis("off")

    for i, layer_name in enumerate(layer_names, start=1):
        heatmap = _activation_to_heatmap(activations[layer_name], out_size=IMG_SIZE)
        axes[i].imshow(img_np)
        axes[i].imshow(heatmap, cmap="jet", alpha=0.45, interpolation="bilinear")
        axes[i].set_title(layer_name, fontsize=9)
        axes[i].axis("off")

    for j in range(n_plots, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()


if "model" not in globals() or model is None:
    recovered_results = []
    for run in training_plan:
        try:
            recovered_results.append(load_run_summary(run["run_id"]))
        except FileNotFoundError:
            pass

    if len(recovered_results) == 0:
        raise RuntimeError("No trained model found. Run at least one model training cell first.")

    best_recovered = max(recovered_results, key=lambda r: r["test_f1"])
    model = CNNWithTextureFusion(num_classes=num_classes, glcm_dim=GLCM_FEATURE_DIM).to(device)
    model.load_state_dict(torch.load(best_recovered["weight_path"], map_location=device))

# Run visualization for 1 AI sample and 1 Real sample from ds_test
model.eval()
label_name_map = _get_label_name_map(ds_test, label_col)

# Prefer explicit binary labels if present; otherwise use first two unique labels
all_labels = sorted(set(int(v) for v in ds_test[label_col]))
if 0 in all_labels and 1 in all_labels:
    labels_to_show = [0, 1]
else:
    labels_to_show = all_labels[:2]

print("Visualizing CNN branch heatmaps for labels:", labels_to_show)
for lbl in labels_to_show:
    sample_idx = _find_first_index_for_label(ds_test, label_col, lbl)
    ex = ds_test[sample_idx]
    img = ex["image"]
    if not isinstance(img, Image.Image):
        img = Image.fromarray(np.asarray(img))

    print(f"\nLabel {lbl} ({label_name_map.get(lbl, lbl)}), sample index: {sample_idx}")
    _plot_layerwise_heatmaps_for_one_sample(model, img, int(lbl), label_name_map)

print("\nDone: displayed layer-wise heatmaps for one sample per class.")